In [7]:
from pathlib import Path
from time import perf_counter

import mlx.core as mx
from docling_core.types.doc import DocTagsDocument, DoclingDocument
from mlx_vlm import load, stream_generate
from mlx_vlm.prompt_utils import apply_chat_template
from mlx_vlm.utils import load_config
from transformers.image_utils import load_image


In [8]:
MODEL_ID = "ibm-granite/granite-docling-258M-mlx"
PROMPT = "Convert this page to docling."
MAX_NEW_TOKENS = 8192

# IBM publishes this quantized MLX checkpoint specifically for Apple Silicon.
model, processor = load(MODEL_ID)
config = load_config(MODEL_ID)

page_paths = sorted(Path("resnet_first_5_pages_png").glob("page_*.png"))
if not page_paths:
    raise FileNotFoundError("No page PNGs found in resnet_first_5_pages_png")

formatted_prompt = apply_chat_template(processor, config, PROMPT, num_images=1)
print(f"Found {len(page_paths)} pages")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Found 5 pages


In [ ]:
# Process each page independently, preserving the document's page order.
started_at = perf_counter()
page_doctags = []
page_images = []
generated_tokens = 0

for page_number, page_path in enumerate(page_paths, start=1):
    doctags = ""
    image = load_image(str(page_path))
    for token in stream_generate(
        model,
        processor,
        formatted_prompt,
        [image],
        max_tokens=MAX_NEW_TOKENS,
        temp=0.0,
        verbose=False,
    ):
        doctags += token.text
        if "</doctag>" in doctags:
            break


    if "</doctag>" not in doctags:
        raise RuntimeError(f"Page {page_number} did not reach the DocTags terminator")
    page_doctags.append(doctags)
    page_images.append(image)
    generated_tokens += token.generation_tokens
    print(f"Page {page_number}/{len(page_paths)}: {len(doctags)} characters")
    break

document_doctags = "\n".join(page_doctags)
elapsed_seconds = perf_counter() - started_at
tokens_per_second = generated_tokens / elapsed_seconds
print(
    f"Generated {generated_tokens} tokens ({tokens_per_second:.2f} tokens/s) and "
    f"{len(document_doctags)} characters across {len(page_doctags)} pages in {elapsed_seconds:.2f}s"
)

# Re-run one causal prefill over [prompt + generated DocTags] to capture the EAGLE-3 taps.
# NOTE: layers must get mask="causal" (the shared extractor does this); passing mask=None
# bypasses create_attention_mask and yields bidirectional attention.
from fastdocling.extract import TraceExtractor
extractor = TraceExtractor(prompt=PROMPT, model=model, processor=processor, config=config)
(_, trace), = extractor.extract([(page_images[-1], page_doctags[-1])], batch_size=1)
mx.save_safetensors("last_hidden_state.safetensors", trace)
print(f"Saved final-page trace: {trace['last_hidden_state'].shape}, taps={extractor.taps}")


In [11]:
doctags_document = DocTagsDocument.from_doctags_and_image_pairs(
    page_doctags, page_images
)
document = DoclingDocument.load_from_doctags(doctags_document)
document_markdown = document.export_to_markdown()
print(document_markdown)

## Deep Residual Learning for Image Recognition

Kaiming He

Xiangyu Zhang

Shaoqing Ren

Jian Sun

Microsoft Research

{ kahe, v-xiangz, v-shren, jiansun } @microsoft.com

## Abstract

Deeper neural networks are more difficult to train. We present a residual learning framework to ease the training of networks that are substantially deeper than those used previously. We explicitly reformulate the layers as learning residual functions with reference to the layer inputs, instead of learning unreferenced functions. We provide comprehensive empirical evidence showing that these residual networks are easier to optimize, and can gain accuracy from considerably increased depth. On the ImageNet dataset we evaluate residual nets with a depth of up to 152 layers-8 × deeper than VGG nets [41] but still having lower complexity. An ensemble of these residual nets achieves 3.57% error on the ImageNet test set. This result won the 1st place on the ILSVRC 2015 classification task. We also present anal